# Modelos Fundacionais para Dados Tabulares: Tutorial Prático com TabPFN

Durante mais de uma década, os algoritmos baseados em árvores de decisão com *gradient boosting* (**XGBoost**, **LightGBM**, **CatBoost**) foram a escolha por omissão para dados tabulares: folhas de cálculo, bases de dados relacionais e registos estruturados.

Os **modelos fundacionais tabulares** (*Tabular Foundation Models*), de que o **TabPFN** é o exemplo mais conhecido, propõem outra forma de trabalhar: em vez de treinar um modelo novo para cada tabela, usa-se um *Transformer* já pré-treinado, que recebe a tabela de treino como contexto e prevê diretamente.

Este notebook compara os dois paradigmas no mesmo problema, com um protocolo de avaliação que permite distinguir uma diferença real de simples ruído de amostragem.

### Como funciona o TabPFN por dentro

1. **Pré-treino em dados sintéticos.** O TabPFN é um *Transformer* pré-treinado *offline* em milhões de conjuntos de dados tabulares sintéticos, gerados a partir de **modelos causais estruturais** (SCM). A primeira versão usava também **redes neuronais bayesianas** como gerador (não confundir com *redes bayesianas*, que são outra coisa). Como cada tabela sintética tem a resposta certa conhecida, o modelo aprende, na prática, um algoritmo geral de previsão em tabelas.
2. **Aprendizagem em contexto (*in-context learning*).** Ao chamar `fit(X_train, y_train)` **não há retropropagação nem atualização de pesos**. O `fit` prepara o pré-processamento interno e guarda o conjunto de treino como contexto, de forma semelhante a um *prompt* num LLM.
3. **Atenção em duas dimensões.** No `predict_proba`, o *Transformer* recebe ao mesmo tempo as linhas de treino (com etiqueta) e as linhas de teste (sem etiqueta), e alterna atenção entre **atributos (colunas)** e **amostras (linhas)** numa única passagem para a frente (*forward pass*). Esta arquitetura existe desde o TabPFN v2.

### O que o notebook faz, passo a passo

| Secção | O que acontece | Para quê |
| :--- | :--- | :--- |
| 1 | Instalação com versão fixa e verificação de hardware | Resultados reprodutíveis |
| 2 | Licença, autenticação e escolha da versão do modelo | Evitar que a descarga dos pesos falhe |
| 3 | Parâmetros da experiência num único sítio | Trocar de dataset sem tocar no resto |
| 4 | Carregamento dos dados e codificação genérica do alvo | Funcionar com qualquer problema binário |
| 5 | Pré-processamento por modelo, dentro de *pipelines* | Tratar bem as variáveis categóricas, sem fuga de informação |
| 6 | Uma partição treino/teste, com tempos separados | Ver os dois paradigmas a funcionar |
| 7 | Incerteza dessa partição (*bootstrap*) | Perceber porque uma partição não chega |
| 8 | Validação cruzada estratificada repetida | Estimativa estável do desempenho |
| 9 | Comparação emparelhada entre modelos | Decidir se a diferença é real |
| 10 | Gráficos | Ler os resultados de relance |
| 11 | Leitura automática dos resultados | Conclusões escritas a partir dos números obtidos |
| 12–14 | Quando usar cada abordagem, próximos passos e referências | Enquadramento |

---
**Execução no Google Colab:** ative a GPU em **`Ambiente de execução > Alterar tipo de ambiente de execução > GPU T4`**. O notebook também corre em CPU, mas o TabPFN fica bastante mais lento; por isso, em CPU o número de repetições da validação cruzada é reduzido automaticamente (secção 3).

## 1. Configuração do ambiente e verificação de hardware

**O que esta célula faz**

- Instala o `tabpfn` numa **versão fixa** (`10.2.0`). O pacote muda o modelo por omissão entre versões, por isso, sem fixar a versão, o mesmo notebook daria resultados diferentes daqui a uns meses.
- Instala o `lightgbm`. As restantes bibliotecas (scikit-learn, pandas, SciPy, Matplotlib) já vêm no Colab.
- Importa tudo o que o notebook usa, num único sítio, para as dependências ficarem visíveis.
- Deteta se há GPU e guarda o resultado em `device`, que é depois passado ao TabPFN.
- Imprime as versões das bibliotecas, para ficarem registadas junto dos resultados.

In [ ]:
!pip install -q "tabpfn==10.2.0" lightgbm

import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import sklearn
import torch
import lightgbm as lgb
import tabpfn

from scipy import stats
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             brier_score_loss, log_loss, roc_auc_score, roc_curve)
from sklearn.model_selection import (RandomizedSearchCV, RepeatedStratifiedKFold,
                                     StratifiedKFold, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler

# Avisos repetitivos das bibliotecas escondem os resultados; os erros continuam a aparecer.
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo selecionado: {device.upper()}")
if device == "cuda":
    print(f"GPU detetada: {torch.cuda.get_device_name(0)}")
else:
    print("Aviso: a executar em CPU. O TabPFN funciona, mas é bastante mais lento do que em GPU.")

print(f"Versões -> tabpfn {tabpfn.__version__} | lightgbm {lgb.__version__} | "
      f"scikit-learn {sklearn.__version__} | pandas {pd.__version__} | scipy {scipy.__version__}")

**Como ler o resultado.** Deve aparecer `CUDA` e o nome da GPU. Se aparecer `CPU`, o notebook continua a funcionar, mas a comparação de tempos deixa de ser representativa do uso normal do TabPFN.

## 2. Licença, autenticação e versão do modelo

Este passo não existe no LightGBM, mas é necessário no TabPFN, porque os pesos do modelo são descarregados da Internet e os das versões recentes têm restrições.

| Pesos | Licença | Precisa de autenticação? |
| :--- | :--- | :--- |
| TabPFN v2 | Aberta (Apache 2.0 com requisito de atribuição) | Não |
| TabPFN 2.5, 2.6, 3 e 3.5 | **Não comercial** (investigação e avaliação interna) | Sim: conta na Prior Labs e aceitação da licença |

O código do pacote `tabpfn` é aberto (Apache 2.0); a restrição aplica-se aos pesos. Para uso comercial das versões recentes é preciso licença própria da Prior Labs.

**Como obter o token (uma única vez)**

1. Crie conta em <https://ux.priorlabs.ai> e aceite a licença do modelo no separador *Licenses*.
2. Copie a chave no separador *API Keys*.
3. No Colab, abra o painel **Secrets** (ícone da chave, à esquerda), crie um segredo chamado `TABPFN_TOKEN` com essa chave e ative o acesso para este notebook.

**O que esta célula faz**

- Procura o token primeiro na variável de ambiente `TABPFN_TOKEN` e depois nos *Secrets* do Colab. O valor nunca é impresso.
- Se encontrar o token, usa o **TabPFN-3.5** (o modelo por omissão desta versão do pacote).
- Se não encontrar, usa o **TabPFN v2**, que não exige autenticação. Assim o notebook corre sempre até ao fim, e fica registado no ecrã qual o modelo usado.

In [ ]:
from tabpfn.constants import ModelVersion

VERSAO_COM_TOKEN = ModelVersion.V3_5   # modelo mais recente; licença não comercial, exige token
VERSAO_SEM_TOKEN = ModelVersion.V2     # pesos abertos; não exige token


def obter_token_tabpfn():
    # 1) Variável de ambiente (útil fora do Colab).
    token = os.environ.get("TABPFN_TOKEN")
    if token:
        return token
    # 2) Secrets do Colab. Fora do Colab, ou sem o segredo definido, isto falha e devolvemos None.
    try:
        from google.colab import userdata
        token = userdata.get("TABPFN_TOKEN")
    except Exception:
        token = None
    if token:
        os.environ["TABPFN_TOKEN"] = token   # é aqui que o pacote tabpfn o vai procurar
    return token


tem_token = bool(obter_token_tabpfn())
VERSAO_TABPFN = VERSAO_COM_TOKEN if tem_token else VERSAO_SEM_TOKEN

if tem_token:
    print(f"Token encontrado. Modelo a usar: TabPFN {VERSAO_TABPFN.value} (licença não comercial).")
else:
    print(f"Token TABPFN_TOKEN não encontrado. Modelo a usar: TabPFN {VERSAO_TABPFN.value} (pesos abertos).")
    print("Para usar o modelo mais recente, siga os três passos descritos acima e volte a correr esta célula.")

**Como ler o resultado.** A linha impressa indica qual o modelo que vai ser avaliado. Tenha isto presente ao interpretar os resultados: o TabPFN v2 e o TabPFN-3.5 são modelos diferentes, e o que se observar com um não se aplica automaticamente ao outro.

## 3. Parâmetros da experiência

**O que esta célula faz.** Junta num único sítio tudo o que se pode querer alterar. As células seguintes leem estes valores e não têm nomes de datasets nem de classes escritos à mão.

| Parâmetro | Significado |
| :--- | :--- |
| `DATASET_NOME`, `DATASET_VERSAO` | Conjunto de dados do OpenML a carregar |
| `CLASSE_POSITIVA` | Etiqueta tratada como classe 1. Com `None`, usa-se a segunda etiqueta por ordem alfabética (em `credit-g`, `good`) |
| `SEMENTE` | Semente aleatória usada em todas as partições e modelos |
| `N_PARTICOES`, `N_REPETICOES` | Validação cruzada: 5 partições, repetidas 3 vezes em GPU (15 avaliações) ou 1 vez em CPU (5 avaliações) |
| `N_ITER_AFINACAO` | Número de combinações de hiperparâmetros testadas na afinação do LightGBM |
| `N_BOOTSTRAP` | Número de reamostragens usadas na secção 7 |

In [ ]:
DATASET_NOME = "credit-g"
DATASET_VERSAO = 1
CLASSE_POSITIVA = None

SEMENTE = 42
N_PARTICOES = 5
N_REPETICOES = 3 if device == "cuda" else 1
N_ITER_AFINACAO = 20
N_BOOTSTRAP = 1000

print(f"Dataset: {DATASET_NOME} (versão {DATASET_VERSAO})")
print(f"Validação cruzada: {N_PARTICOES} partições x {N_REPETICOES} repetição(ões) "
      f"= {N_PARTICOES * N_REPETICOES} avaliações por modelo")

## 4. Carregamento dos dados e codificação do alvo

Usamos o **`credit-g`** (*German Credit Data*) do OpenML: **1 000 registos** e **20 variáveis** (7 numéricas e 13 categóricas) para classificar o risco de crédito em `good` ou `bad`. É um conjunto pequeno, a zona onde os modelos fundacionais tabulares foram pensados para funcionar melhor.

**O que esta célula faz**

1. Descarrega o dataset com `fetch_openml` e separa atributos (`X`) e alvo.
2. **Codifica o alvo de forma genérica.** Em vez de comparar com a palavra `"good"`, o código descobre as duas etiquetas existentes e atribui 1 à classe positiva. Isto permite trocar de dataset na secção 3 sem partir o notebook. Se o problema não for binário, a célula pára com uma mensagem clara.
3. **Identifica as colunas categóricas** (todas as que não são numéricas) e converte-as para o tipo `category` do pandas. Esta informação é usada na secção 5 para cada modelo as tratar da forma certa.
4. Calcula a proporção da classe maioritária, que serve de referência para a *accuracy*.

In [ ]:
dataset = fetch_openml(name=DATASET_NOME, version=DATASET_VERSAO, as_frame=True)
X = dataset.data.copy()
alvo_original = dataset.target.astype(str)

# --- Alvo: de etiquetas de texto para 0/1, sem depender de um nome de classe fixo ---
codificador_alvo = LabelEncoder().fit(alvo_original)
classes = list(codificador_alvo.classes_)
if len(classes) != 2:
    raise ValueError(f"Este notebook trata classificação binária; '{DATASET_NOME}' tem {len(classes)} classes: {classes}")

classe_positiva = CLASSE_POSITIVA if CLASSE_POSITIVA is not None else classes[1]
if classe_positiva not in classes:
    raise ValueError(f"CLASSE_POSITIVA='{classe_positiva}' não existe no alvo. Classes disponíveis: {classes}")
classe_negativa = [c for c in classes if c != classe_positiva][0]
y = (alvo_original == classe_positiva).astype(int).to_numpy()

# --- Atributos: separar numéricos de categóricos ---
colunas_cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
colunas_num = [c for c in X.columns if c not in colunas_cat]
for c in colunas_cat:
    X[c] = X[c].astype("category")

proporcao_positiva = y.mean()
referencia_accuracy = max(proporcao_positiva, 1 - proporcao_positiva)

print(f"Dimensões: {X.shape[0]} amostras x {X.shape[1]} atributos "
      f"({len(colunas_num)} numéricos, {len(colunas_cat)} categóricos)")
print(f"Classe positiva (1): '{classe_positiva}' -> {proporcao_positiva:.1%} | "
      f"classe negativa (0): '{classe_negativa}' -> {1 - proporcao_positiva:.1%}")
print(f"Valores em falta: {int(X.isna().sum().sum())}")
print(f"Accuracy de quem prevê sempre a classe maioritária: {referencia_accuracy:.3f}")
X.head(5)

**Como ler o resultado.** Em `credit-g`, cerca de 70 % dos registos são `good`. Um modelo que responda sempre `good` acerta 70 % das vezes sem ter aprendido nada, por isso uma *accuracy* de 0,72 é fraca, apesar de parecer razoável. É a razão para as métricas principais deste notebook serem outras (secção 6).

## 5. Pré-processamento e definição dos modelos

A versão anterior deste notebook aplicava a mesma codificação ordinal a tudo. O problema é que, depois de codificada, uma variável como `purpose` (finalidade do crédito) passa a ser um número de 0 a 9 por ordem alfabética, e os modelos tratam-na como se essa ordem tivesse significado. Aqui, cada modelo recebe as variáveis categóricas da forma que sabe aproveitar.

| Modelo | Tratamento das categóricas | Tratamento das numéricas |
| :--- | :--- | :--- |
| Referência (classe maioritária) | Ignora os atributos | Ignora os atributos |
| Regressão logística | *One-hot encoding* | Imputação pela mediana e padronização |
| LightGBM (omissão e afinado) | Suporte nativo: recebe as colunas com tipo `category` | Sem transformação |
| TabPFN | Códigos ordinais **e** indicação de quais são as colunas categóricas (`categorical_features_indices`) | Sem transformação |

**Porque é que tudo está dentro de *pipelines*.** Os codificadores aprendem com os dados (por exemplo, a lista de categorias). Ao ficarem dentro de um `Pipeline`, são ajustados apenas no conjunto de treino de cada partição, e as linhas de teste nunca influenciam a preparação dos dados. Evita-se assim a fuga de informação (*data leakage*).

**Os cinco modelos**

- **Referência:** prevê sempre a proporção das classes. Fixa o mínimo que qualquer modelo tem de ultrapassar.
- **Regressão logística:** modelo linear simples. Se um modelo complexo não a ultrapassa com clareza, a complexidade não se justifica.
- **LightGBM (omissão):** hiperparâmetros de fábrica (100 árvores com 31 folhas). Num conjunto de 800 linhas tende a sobreajustar.
- **LightGBM (afinado):** pesquisa aleatória de hiperparâmetros com validação cruzada interna de 3 partições. É a comparação justa com o TabPFN, porque é assim que as árvores são usadas na prática. O tempo de `fit` deste modelo inclui toda a pesquisa.
- **TabPFN:** sem afinação. É essa a proposta do modelo.

**O que esta célula faz.** Define a função `criar_modelos()`, que devolve um dicionário com os cinco modelos acabados de criar (não treinados). Define também as cores usadas nos gráficos, fixas por modelo.

In [ ]:
# Espaço de pesquisa do LightGBM: árvores mais pequenas e mais regularização do que os valores de fábrica.
ESPACO_LGBM = {
    "n_estimators": stats.randint(50, 500),
    "learning_rate": stats.loguniform(0.01, 0.2),
    "num_leaves": stats.randint(4, 32),
    "min_child_samples": stats.randint(5, 40),
    "subsample": stats.uniform(0.6, 0.4),          # valores entre 0,6 e 1,0
    "colsample_bytree": stats.uniform(0.5, 0.5),   # valores entre 0,5 e 1,0
    "reg_lambda": stats.loguniform(0.01, 10),
    "min_data_per_group": stats.randint(10, 100),  # controla a divisão por categorias
    "cat_smooth": stats.loguniform(1, 50),
}


def criar_modelos():
    # --- Regressão logística: one-hot nas categóricas, padronização nas numéricas ---
    prep_linear = ColumnTransformer([
        ("num", Pipeline([("imputar", SimpleImputer(strategy="median")),
                          ("escalar", StandardScaler())]), colunas_num),
        ("cat", Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), colunas_cat),
    ])
    reg_logistica = Pipeline([("prep", prep_linear),
                              ("modelo", LogisticRegression(max_iter=5000))])

    # --- LightGBM: lê diretamente as colunas de tipo 'category' ---
    lgbm_omissao = lgb.LGBMClassifier(random_state=SEMENTE, verbose=-1)

    lgbm_afinado = RandomizedSearchCV(
        estimator=lgb.LGBMClassifier(random_state=SEMENTE, verbose=-1, subsample_freq=1, n_jobs=1),
        param_distributions=ESPACO_LGBM,
        n_iter=N_ITER_AFINACAO,
        scoring="neg_log_loss",   # otimiza a qualidade das probabilidades, não só a ordenação
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEMENTE),
        random_state=SEMENTE,
        n_jobs=-1,
        refit=True,               # no fim, volta a treinar com a melhor combinação
    )

    # --- TabPFN: códigos ordinais + índices das colunas categóricas ---
    # O ColumnTransformer põe primeiro as categóricas, logo os seus índices são 0, 1, ..., k-1.
    prep_tabpfn = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), colunas_cat),
        ("num", "passthrough", colunas_num),
    ])
    modelo_tabpfn = TabPFNClassifier.create_default_for_version(
        VERSAO_TABPFN,
        device=device,
        random_state=SEMENTE,
        categorical_features_indices=list(range(len(colunas_cat))),
    )
    pipeline_tabpfn = Pipeline([("prep", prep_tabpfn), ("modelo", modelo_tabpfn)])

    return {
        "Referência (classe maioritária)": DummyClassifier(strategy="prior"),
        "Regressão logística": reg_logistica,
        "LightGBM (omissão)": lgbm_omissao,
        "LightGBM (afinado)": lgbm_afinado,
        "TabPFN": pipeline_tabpfn,
    }


from tabpfn import TabPFNClassifier

NOME_TABPFN = "TabPFN"
NOME_RIVAL = "LightGBM (afinado)"   # adversário principal na comparação

# Cores fixas por modelo (paleta verificada para daltonismo). A referência não entra nos gráficos.
CORES = {
    "TabPFN": "#257de0",
    "LightGBM (afinado)": "#f47230",
    "LightGBM (omissão)": "#00b894",
    "Regressão logística": "#f2a600",
}
MODELOS_GRAFICO = list(CORES)

print("Modelos definidos:")
for nome in criar_modelos():
    print(f"  - {nome}")

## 6. Uma partição treino/teste: os dois paradigmas a funcionar

Antes da avaliação completa, vale a pena ver cada modelo a trabalhar uma vez, com os tempos de `fit` e de `predict` medidos em separado. É aqui que a diferença conceptual fica visível:

- **LightGBM:** o `fit` é onde o trabalho acontece (construção das árvores); o `predict` é quase instantâneo.
- **TabPFN:** o `fit` prepara os dados e guarda o contexto; o cálculo pesado acontece no `predict_proba`, que processa treino e teste em conjunto.

**O que esta célula faz**

1. Divide os dados em 80 % para treino e 20 % para teste, mantendo a proporção das classes (`stratify=y`).
2. **Faz um aquecimento do TabPFN** com 50 linhas. A primeira utilização descarrega os pesos e inicializa a GPU, o que pode demorar dezenas de segundos. Sem este passo, esse custo único seria contabilizado como tempo do modelo.
3. Para cada modelo, mede o tempo de `fit` e o tempo de `predict_proba` separadamente.
4. Obtém as classes a partir das probabilidades (limiar 0,5), em vez de chamar `predict`. No TabPFN, `predict` repete internamente a passagem completa pelo *Transformer*, pelo que chamá-lo duplicaria o tempo de inferência.
5. Calcula cinco métricas.

**As métricas**

| Métrica | O que mede | Melhor quando |
| :--- | :--- | :--- |
| ROC-AUC | Capacidade de ordenar: probabilidade de um caso positivo receber pontuação mais alta do que um negativo. Não depende do limiar | Mais alto (0,5 = acaso) |
| *Log loss* | Qualidade das probabilidades; penaliza muito a confiança errada | Mais baixo |
| Brier | Erro quadrático médio das probabilidades | Mais baixo |
| *Accuracy* | Proporção de acertos com limiar 0,5 | Mais alto, mas comparar com a classe maioritária |
| *Accuracy* equilibrada | Média da taxa de acerto em cada classe | Mais alto (0,5 = acaso) |

Em risco de crédito interessam sobretudo as três primeiras, porque a decisão final costuma depender de uma probabilidade bem calibrada e de um limiar escolhido em função dos custos, e não do limiar 0,5.

In [ ]:
def calcular_metricas(y_real, proba):
    pred = (proba >= 0.5).astype(int)
    return {
        "ROC-AUC": roc_auc_score(y_real, proba),
        "Log loss": log_loss(y_real, proba, labels=[0, 1]),
        "Brier": brier_score_loss(y_real, proba),
        "Accuracy": accuracy_score(y_real, pred),
        "Accuracy equilibrada": balanced_accuracy_score(y_real, pred),
    }


def avaliar(modelo, X_tr, y_tr, X_te, y_te):
    # Mede fit e predict em separado e devolve métricas, tempos e probabilidades.
    t0 = time.perf_counter()
    modelo.fit(X_tr, y_tr)
    t_fit = time.perf_counter() - t0

    t0 = time.perf_counter()
    proba = modelo.predict_proba(X_te)[:, 1]   # coluna 1 = probabilidade da classe positiva
    t_pred = time.perf_counter() - t0

    resultado = calcular_metricas(y_te, proba)
    resultado["Fit (s)"] = t_fit
    resultado["Predict (s)"] = t_pred
    return resultado, proba


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEMENTE, stratify=y
)
print(f"Treino (contexto): {len(X_train)} linhas | Teste: {len(X_test)} linhas")

# Aquecimento: descarrega os pesos e inicializa a GPU fora das medições de tempo.
t0 = time.perf_counter()
aquecimento = criar_modelos()[NOME_TABPFN]
aquecimento.fit(X_train.iloc[:50], y_train[:50])
aquecimento.predict_proba(X_test.iloc[:5])
print(f"Aquecimento do TabPFN (descarga de pesos + arranque): {time.perf_counter() - t0:.1f}s, fora das medições")

resultados_holdout = {}
probas_holdout = {}
for nome, modelo in criar_modelos().items():
    resultados_holdout[nome], probas_holdout[nome] = avaliar(modelo, X_train, y_train, X_test, y_test)

tabela_holdout = pd.DataFrame(resultados_holdout).T
tabela_holdout["Predict (ms/linha)"] = 1000 * tabela_holdout["Predict (s)"] / len(X_test)
tabela_holdout.round(4)

**Como ler o resultado**

- A linha **Referência** tem ROC-AUC de 0,5 e *accuracy* igual à proporção da classe maioritária. É o ponto de partida.
- Compare as colunas **Fit (s)** e **Predict (s)** entre o LightGBM por omissão e o TabPFN: no primeiro, o tempo está no `fit`; no segundo, está no `predict`.
- O **Fit (s)** do LightGBM afinado inclui dezenas de treinos (a pesquisa de hiperparâmetros). É o custo real de usar árvores bem afinadas.
- **Não tire ainda conclusões sobre qual o melhor modelo.** O conjunto de teste é pequeno e a secção seguinte mostra quanta incerteza existe nestes números.

## 7. Quanta confiança merece uma única partição?

Com um conjunto de teste de cerca de 200 linhas, o ROC-AUC é uma estimativa com bastante ruído: outras 200 linhas dariam outro valor. Para quantificar esse ruído usamos ***bootstrap***.

**O que esta célula faz**

1. Reamostra o conjunto de teste com reposição `N_BOOTSTRAP` vezes (cada reamostragem tem o mesmo tamanho do teste original).
2. Em cada reamostragem, recalcula o ROC-AUC de cada modelo **nas mesmas linhas**. Como as linhas são as mesmas, a comparação é emparelhada.
3. Apresenta o intervalo de confiança a 95 % (percentis 2,5 e 97,5) do ROC-AUC de cada modelo.
4. Faz o mesmo para a **diferença** entre o TabPFN e o LightGBM afinado.

Esta análise mede apenas a incerteza devida ao tamanho do conjunto de teste. Não inclui a incerteza de ter escolhido estas 800 linhas de treino e não outras; essa é tratada na secção 8.

In [ ]:
rng = np.random.default_rng(SEMENTE)
n_teste = len(y_test)
auc_boot = {nome: [] for nome in MODELOS_GRAFICO}

for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n_teste, n_teste)
    if y_test[idx].min() == y_test[idx].max():
        continue   # reamostragem com uma só classe: o AUC não está definido
    for nome in MODELOS_GRAFICO:
        auc_boot[nome].append(roc_auc_score(y_test[idx], probas_holdout[nome][idx]))

auc_boot = pd.DataFrame(auc_boot)
tabela_ic = pd.DataFrame({
    "ROC-AUC observado": [resultados_holdout[m]["ROC-AUC"] for m in MODELOS_GRAFICO],
    "IC 95% inferior": auc_boot.quantile(0.025).values,
    "IC 95% superior": auc_boot.quantile(0.975).values,
}, index=MODELOS_GRAFICO)
tabela_ic["Largura do IC"] = tabela_ic["IC 95% superior"] - tabela_ic["IC 95% inferior"]

dif_boot = auc_boot[NOME_TABPFN] - auc_boot[NOME_RIVAL]
dif_obs = resultados_holdout[NOME_TABPFN]["ROC-AUC"] - resultados_holdout[NOME_RIVAL]["ROC-AUC"]
ic_dif = dif_boot.quantile([0.025, 0.975]).values

print(f"Diferença de ROC-AUC ({NOME_TABPFN} menos {NOME_RIVAL}): {dif_obs:+.4f} "
      f"| IC 95%: [{ic_dif[0]:+.4f}, {ic_dif[1]:+.4f}]")
if ic_dif[0] <= 0 <= ic_dif[1]:
    print("O intervalo inclui zero: esta partição, sozinha, não permite dizer qual dos dois é melhor.")
else:
    print("O intervalo não inclui zero nesta partição. Falta confirmar com a validação cruzada (secção 8).")
tabela_ic.round(4)

**Como ler o resultado**

- A coluna **Largura do IC** costuma rondar 0,10 a 0,15 neste dataset. Diferenças de 0,01 ou 0,02 entre modelos ficam muito abaixo dessa largura.
- Se o intervalo da diferença **inclui zero**, a partição não distingue os dois modelos. É o resultado mais habitual com 200 linhas de teste.
- É por isto que a comparação não deve assentar numa única partição: seria possível obter o resultado contrário apenas mudando a semente.

## 8. Validação cruzada estratificada repetida

Para uma estimativa estável, cada modelo é avaliado várias vezes, em partições diferentes dos dados.

**Como funciona**

- Os dados são divididos em `N_PARTICOES` blocos com a mesma proporção de classes. Cada bloco serve uma vez de teste, com os restantes a servir de treino.
- O processo é repetido `N_REPETICOES` vezes, com divisões aleatórias diferentes.
- Todos os modelos veem **exatamente as mesmas partições**, o que permite comparar resultados par a par na secção 9.

**O que esta célula faz**

1. Cria as partições com `RepeatedStratifiedKFold`.
2. Em cada partição, cria modelos novos (`criar_modelos()`), treina-os e avalia-os com a função `avaliar` da secção 6. O pré-processamento e a afinação do LightGBM são refeitos dentro de cada partição, usando apenas os dados de treino dessa partição.
3. Guarda uma linha por modelo e por partição na tabela `cv`.
4. Apresenta a média e o desvio padrão de cada métrica.

Esta é a célula mais demorada do notebook: alguns minutos em GPU, mais em CPU.

In [ ]:
particoes = RepeatedStratifiedKFold(n_splits=N_PARTICOES, n_repeats=N_REPETICOES, random_state=SEMENTE)
total = N_PARTICOES * N_REPETICOES
registos = []

t_inicio = time.perf_counter()
for i, (idx_tr, idx_te) in enumerate(particoes.split(X, y), start=1):
    X_tr, X_te = X.iloc[idx_tr], X.iloc[idx_te]
    y_tr, y_te = y[idx_tr], y[idx_te]
    for nome, modelo in criar_modelos().items():
        resultado, _ = avaliar(modelo, X_tr, y_tr, X_te, y_te)
        resultado.update({"Modelo": nome, "Partição": i})
        registos.append(resultado)
    print(f"Partição {i}/{total} concluída ({time.perf_counter() - t_inicio:.0f}s acumulados)")

cv = pd.DataFrame(registos)
METRICAS = ["ROC-AUC", "Log loss", "Brier", "Accuracy", "Accuracy equilibrada"]
ORDEM = list(criar_modelos())

media = cv.groupby("Modelo")[METRICAS + ["Fit (s)", "Predict (s)"]].mean().loc[ORDEM]
desvio = cv.groupby("Modelo")[METRICAS].std().loc[ORDEM]

resumo = pd.DataFrame(index=ORDEM)
for m in METRICAS:
    resumo[m] = [f"{media.loc[n, m]:.4f} ± {desvio.loc[n, m]:.4f}" for n in ORDEM]
resumo["Fit (s)"] = media["Fit (s)"].round(3)
resumo["Predict (s)"] = media["Predict (s)"].round(3)
print(f"\nMédia ± desvio padrão em {total} avaliações por modelo:")
resumo

**Como ler o resultado**

- Cada célula mostra **média ± desvio padrão** entre partições. O desvio padrão indica quanto o resultado oscila consoante os dados que calham no teste.
- Compare o desvio padrão do ROC-AUC com a diferença entre as médias de dois modelos. Se a diferença for muito menor do que o desvio, é preciso o teste emparelhado da secção seguinte para decidir.
- A diferença entre **LightGBM (omissão)** e **LightGBM (afinado)** mostra quanto vale a afinação neste dataset. É essa a parcela de trabalho que o TabPFN dispensa.

## 9. A diferença entre modelos é real?

Como todos os modelos foram avaliados nas mesmas partições, podemos calcular a diferença **partição a partição**. Isto elimina grande parte do ruído, porque uma partição difícil é difícil para todos.

**O que esta célula faz.** Para o TabPFN contra cada um dos outros modelos, e para ROC-AUC e *log loss*, calcula:

- **Diferença média** (TabPFN menos o outro modelo). No ROC-AUC, positivo favorece o TabPFN; no *log loss*, negativo favorece o TabPFN.
- **Vitórias:** em quantas partições o TabPFN foi melhor.
- **Valor p** de um **teste t corrigido para reamostragem** (Nadeau e Bengio).

**Porquê um teste corrigido.** Na validação cruzada, os conjuntos de treino de partições diferentes partilham a maior parte das linhas, pelo que os resultados não são independentes. Um teste t normal ignora essa dependência e declara diferenças significativas com demasiada facilidade. A correção aumenta a variância em função da proporção entre teste e treino:

$$t = \frac{\bar{d}}{\sqrt{\left(\frac{1}{J} + \frac{n_{teste}}{n_{treino}}\right) s_d^2}}$$

em que $\bar{d}$ é a diferença média, $s_d^2$ a variância das diferenças e $J$ o número de avaliações.

In [ ]:
def teste_t_corrigido(diferencas, racio_teste_treino):
    # Teste t corrigido para reamostragem (Nadeau e Bengio). Devolve estatística t e valor p bilateral.
    d = np.asarray(diferencas, dtype=float)
    J = len(d)
    variancia = d.var(ddof=1)
    if J < 2 or variancia == 0:
        return np.nan, np.nan
    t = d.mean() / np.sqrt((1 / J + racio_teste_treino) * variancia)
    p = 2 * stats.t.sf(abs(t), df=J - 1)
    return t, p


racio = 1 / (N_PARTICOES - 1)   # n_teste / n_treino numa validação cruzada de k partições
por_particao = {m: cv.pivot(index="Partição", columns="Modelo", values=m) for m in ["ROC-AUC", "Log loss"]}

linhas = []
for outro in [n for n in MODELOS_GRAFICO if n != NOME_TABPFN]:
    for metrica, maior_e_melhor in [("ROC-AUC", True), ("Log loss", False)]:
        d = por_particao[metrica][NOME_TABPFN] - por_particao[metrica][outro]
        vitorias = int((d > 0).sum()) if maior_e_melhor else int((d < 0).sum())
        t, p = teste_t_corrigido(d, racio)
        linhas.append({
            "TabPFN contra": outro,
            "Métrica": metrica,
            "Diferença média": d.mean(),
            "Desvio padrão": d.std(ddof=1),
            "Vitórias do TabPFN": f"{vitorias}/{len(d)}",
            "Valor p": p,
        })

comparacao = pd.DataFrame(linhas).set_index(["TabPFN contra", "Métrica"])
comparacao.round(4)

**Como ler o resultado**

- Um **valor p abaixo de 0,05** indica que a diferença dificilmente se deve ao acaso das partições. Acima disso, os dados não permitem separar os dois modelos, o que é diferente de provar que são iguais.
- As **vitórias** ajudam a perceber a consistência: ganhar em 14 de 15 partições por pouco é mais convincente do que ganhar em 8 de 15 por muito.
- Com poucas avaliações (por exemplo, 5 em CPU), o teste tem pouca potência e só deteta diferenças grandes. Aumentar `N_REPETICOES` na secção 3 dá mais resolução.
- Estes resultados dizem respeito a **este dataset**. Não se generalizam a outros domínios sem repetir a experiência.

## 10. Visualização dos resultados

Três figuras, cada uma a responder a uma pergunta.

**Figura 1: que modelo prevê melhor, e com que estabilidade?** Mostra o ROC-AUC e o *log loss* da validação cruzada. Cada ponto pequeno é uma partição; o ponto grande é a média, e a barra horizontal é a média ± um desvio padrão. As duas métricas têm eixos próprios, porque estão em escalas diferentes.

In [ ]:
COR_TEXTO = "#191919"
COR_SECUNDARIA = "#666666"
COR_GRELHA = "#e6e6e6"


def limpar_eixos(ax):
    # Eixos e grelha discretos, para os dados ficarem em primeiro plano.
    for lado in ["top", "right"]:
        ax.spines[lado].set_visible(False)
    for lado in ["left", "bottom"]:
        ax.spines[lado].set_color(COR_GRELHA)
    ax.tick_params(colors=COR_SECUNDARIA, length=0)
    ax.set_axisbelow(True)


fig, eixos = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
posicoes = {nome: len(MODELOS_GRAFICO) - 1 - i for i, nome in enumerate(MODELOS_GRAFICO)}
desloc = np.random.default_rng(SEMENTE).uniform(-0.16, 0.16, size=total)

for ax, (metrica, nota) in zip(eixos, [("ROC-AUC", "mais alto é melhor"), ("Log loss", "mais baixo é melhor")]):
    for nome in MODELOS_GRAFICO:
        valores = cv.loc[cv["Modelo"] == nome, metrica].to_numpy()
        yb = posicoes[nome]
        ax.scatter(valores, yb + desloc[:len(valores)], s=22, color=CORES[nome], alpha=0.35, linewidths=0)
        ax.errorbar(valores.mean(), yb, xerr=valores.std(ddof=1) if len(valores) > 1 else 0,
                    fmt="o", color=CORES[nome], markersize=9, markeredgecolor="white",
                    markeredgewidth=2, elinewidth=2, capsize=0, zorder=3)
        ax.annotate(f"{valores.mean():.3f}", (valores.mean(), yb), xytext=(0, 12),
                    textcoords="offset points", ha="center", fontsize=10, color=COR_TEXTO, fontweight="bold")
    ax.set_title(f"{metrica} ({nota})", loc="left", fontsize=12, fontweight="bold", color=COR_TEXTO)
    ax.set_yticks(list(posicoes.values()))
    ax.set_yticklabels(list(posicoes.keys()), fontsize=11, color=COR_TEXTO)
    ax.set_ylim(-0.6, len(MODELOS_GRAFICO) - 0.3)
    ax.grid(axis="x", color=COR_GRELHA, linewidth=1)
    limpar_eixos(ax)

fig.suptitle(f"Validação cruzada em {DATASET_NOME}: {total} avaliações por modelo (ponto grande = média; barra = ± 1 desvio padrão)",
             x=0.01, ha="left", fontsize=11, color=COR_SECUNDARIA)
plt.tight_layout()
plt.show()

**Como ler a Figura 1.** Se as nuvens de pontos de dois modelos se sobrepõem quase por completo, a diferença entre as médias é pequena face à variação entre partições. Um modelo claramente melhor aparece deslocado para a direita no ROC-AUC e para a esquerda no *log loss*.

---

**Figura 2: como se comportam as probabilidades?** Usa as previsões da partição única da secção 6.

- **Esquerda, curva ROC:** para cada limiar de decisão, a taxa de verdadeiros positivos contra a de falsos positivos. Quanto mais a curva se aproxima do canto superior esquerdo, melhor. A diagonal corresponde ao acaso.
- **Direita, curva de calibração:** agrupa as previsões em 5 intervalos com o mesmo número de casos e compara a probabilidade média prevista com a proporção real de positivos. Um modelo bem calibrado fica sobre a diagonal: quando prevê 80 %, acerta 80 % das vezes.

In [ ]:
fig, (ax_roc, ax_cal) = plt.subplots(1, 2, figsize=(13, 4.8))

for nome in MODELOS_GRAFICO:
    fpr, tpr, _ = roc_curve(y_test, probas_holdout[nome])
    ax_roc.plot(fpr, tpr, color=CORES[nome], linewidth=2,
                label=f"{nome} (AUC {resultados_holdout[nome]['ROC-AUC']:.3f})")

    prop_real, prob_prevista = calibration_curve(y_test, probas_holdout[nome], n_bins=5, strategy="quantile")
    ax_cal.plot(prob_prevista, prop_real, color=CORES[nome], linewidth=2, marker="o", markersize=8,
                markeredgecolor="white", markeredgewidth=1.5,
                label=f"{nome} (Brier {resultados_holdout[nome]['Brier']:.3f})")

for ax in (ax_roc, ax_cal):
    ax.plot([0, 1], [0, 1], color=COR_SECUNDARIA, linestyle="--", linewidth=1)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.grid(color=COR_GRELHA, linewidth=1)
    limpar_eixos(ax)

ax_roc.set_title("Curva ROC (partição única)", loc="left", fontsize=12, fontweight="bold", color=COR_TEXTO)
ax_roc.set_xlabel("Taxa de falsos positivos", color=COR_SECUNDARIA)
ax_roc.set_ylabel("Taxa de verdadeiros positivos", color=COR_SECUNDARIA)
ax_roc.text(0.62, 0.56, "acaso", color=COR_SECUNDARIA, fontsize=9, rotation=38)
ax_roc.legend(loc="lower right", frameon=False, fontsize=10, labelcolor=COR_TEXTO)

ax_cal.set_title("Curva de calibração (partição única)", loc="left", fontsize=12, fontweight="bold", color=COR_TEXTO)
ax_cal.set_xlabel(f"Probabilidade prevista de '{classe_positiva}'", color=COR_SECUNDARIA)
ax_cal.set_ylabel(f"Proporção real de '{classe_positiva}'", color=COR_SECUNDARIA)
ax_cal.text(0.03, 0.09, "calibração perfeita", color=COR_SECUNDARIA, fontsize=9, rotation=38)
ax_cal.legend(loc="lower right", frameon=False, fontsize=10, labelcolor=COR_TEXTO)

plt.tight_layout()
plt.show()

**Como ler a Figura 2.** As curvas vêm de cerca de 200 linhas de teste, por isso são irregulares, em especial a de calibração (cada ponto resume umas 40 linhas). Servem para ver o comportamento geral: curvas de calibração muito afastadas da diagonal indicam excesso ou falta de confiança. A comparação quantitativa da qualidade das probabilidades faz-se com o *log loss* e o Brier da secção 8.

---

**Figura 3: quanto custa cada modelo em tempo?** Tempo médio de `fit` e de `predict` na validação cruzada, em painéis separados, porque as ordens de grandeza são muito diferentes. O `predict` refere-se a um bloco de teste inteiro (cerca de 200 linhas).

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 3.4), sharey=True)
n_linhas_teste = len(X) // N_PARTICOES

for ax, (coluna, titulo) in zip(eixos, [("Fit (s)", "Tempo médio de fit (s)"),
                                        ("Predict (s)", f"Tempo médio de predict para ~{n_linhas_teste} linhas (s)")]):
    valores = [media.loc[nome, coluna] for nome in MODELOS_GRAFICO]
    barras = ax.barh([posicoes[n] for n in MODELOS_GRAFICO], valores, height=0.55,
                     color=[CORES[n] for n in MODELOS_GRAFICO])
    for barra, v in zip(barras, valores):
        ax.text(v + max(valores) * 0.015, barra.get_y() + barra.get_height() / 2,
                f"{v:.3f}s" if v < 10 else f"{v:.1f}s", va="center", fontsize=10, color=COR_TEXTO)
    ax.set_xlim(0, max(valores) * 1.18)
    ax.set_title(titulo, loc="left", fontsize=12, fontweight="bold", color=COR_TEXTO)
    ax.set_yticks(list(posicoes.values()))
    ax.set_yticklabels(list(posicoes.keys()), fontsize=11, color=COR_TEXTO)
    ax.grid(axis="x", color=COR_GRELHA, linewidth=1)
    limpar_eixos(ax)

fig.suptitle(f"Dispositivo: {device.upper()}. O fit do LightGBM afinado inclui a pesquisa de {N_ITER_AFINACAO} combinações de hiperparâmetros.",
             x=0.01, ha="left", fontsize=11, color=COR_SECUNDARIA)
plt.tight_layout()
plt.show()

**Como ler a Figura 3.** O painel da esquerda mostra o custo de preparar o modelo; o da direita, o custo de o usar. Em produção, o `fit` paga-se uma vez e o `predict` paga-se em cada pedido, por isso o painel da direita é normalmente o que mais pesa na decisão. Os tempos dependem do hardware: em CPU, o `predict` do TabPFN sobe muito.

## 11. Leitura dos resultados

**O que esta célula faz.** Escreve as conclusões a partir dos números obtidos nesta execução, em vez de as fixar no texto do notebook. Se mudar de dataset, de versão do modelo ou de hardware, o texto impresso acompanha os novos resultados.

A regra de decisão é a da secção 9: só se declara que um modelo é melhor quando o valor p do teste corrigido fica abaixo de 0,05.

In [ ]:
def frase_comparacao(outro, metrica, maior_e_melhor):
    linha = comparacao.loc[(outro, metrica)]
    d, p, vitorias = linha["Diferença média"], linha["Valor p"], linha["Vitórias do TabPFN"]
    tabpfn_melhor = d > 0 if maior_e_melhor else d < 0
    if np.isnan(p):
        veredicto = "não foi possível calcular o teste"
    elif p < 0.05:
        txt_p = "p < 0.001" if p < 0.001 else f"p = {p:.3f}"
        veredicto = f"diferença consistente a favor de {'TabPFN' if tabpfn_melhor else outro} ({txt_p})"
    else:
        veredicto = f"os dados não permitem separar os dois modelos (p = {p:.3f})"
    return f"  - {metrica}: diferença média {d:+.4f}, TabPFN melhor em {vitorias} partições -> {veredicto}."


melhor_auc = media.loc[MODELOS_GRAFICO, "ROC-AUC"].idxmax()
melhor_ll = media.loc[MODELOS_GRAFICO, "Log loss"].idxmin()

print(f"Dataset: {DATASET_NOME} | Modelo: TabPFN {VERSAO_TABPFN.value} | Dispositivo: {device.upper()} | {total} avaliações por modelo\n")
print(f"1) Melhor ROC-AUC médio: {melhor_auc} ({media.loc[melhor_auc, 'ROC-AUC']:.4f}). "
      f"Melhor log loss médio: {melhor_ll} ({media.loc[melhor_ll, 'Log loss']:.4f}).\n")

print(f"2) TabPFN contra {NOME_RIVAL} (a comparação justa):")
print(frase_comparacao(NOME_RIVAL, "ROC-AUC", True))
print(frase_comparacao(NOME_RIVAL, "Log loss", False))

print("\n3) TabPFN contra Regressão logística (justifica-se a complexidade?):")
print(frase_comparacao("Regressão logística", "ROC-AUC", True))
print(frase_comparacao("Regressão logística", "Log loss", False))

ganho_afinacao = media.loc["LightGBM (afinado)", "ROC-AUC"] - media.loc["LightGBM (omissão)", "ROC-AUC"]
print(f"\n4) Valor da afinação do LightGBM: {ganho_afinacao:+.4f} de ROC-AUC face aos valores de fábrica, "
      f"com um fit {media.loc['LightGBM (afinado)', 'Fit (s)'] / max(media.loc['LightGBM (omissão)', 'Fit (s)'], 1e-9):.0f}x mais demorado.")

racio_pred = media.loc[NOME_TABPFN, "Predict (s)"] / max(media.loc["LightGBM (omissão)", "Predict (s)"], 1e-9)
print(f"\n5) Custo de inferência: o predict do TabPFN demorou {media.loc[NOME_TABPFN, 'Predict (s)']:.3f}s por bloco de teste, "
      f"{racio_pred:.1f}x o do LightGBM ({media.loc['LightGBM (omissão)', 'Predict (s)']:.4f}s).")
print(f"   Preparar o TabPFN (fit) demorou {media.loc[NOME_TABPFN, 'Fit (s)']:.3f}s, contra "
      f"{media.loc[NOME_RIVAL, 'Fit (s)']:.1f}s do LightGBM afinado.")

**Como enquadrar estas conclusões**

- O que foi impresso acima é o resultado **desta experiência**: um dataset, uma versão do modelo, um orçamento de afinação de 20 combinações para o LightGBM.
- Os resultados publicados em *benchmarks* são outra fonte de evidência, com dezenas de datasets. O relatório do TabPFN-2.5 indica que o modelo, sem afinação, supera árvores afinadas no TabArena. É um resultado reportado pela equipa que desenvolve o modelo e deve ser lido como tal.
- As duas fontes podem não coincidir. Num dataset pequeno e com muito ruído como o `credit-g`, é comum vários modelos ficarem estatisticamente empatados, incluindo a regressão logística.
- Mesmo com desempenho empatado, o TabPFN chega a esse resultado sem afinação, o que reduz o trabalho de modelação. Em contrapartida, tem inferência mais lenta, depende de GPU e os pesos recentes têm licença não comercial.

## 12. Quando usar cada abordagem

| Critério | Modelos fundacionais tabulares (TabPFN) | Árvores com *gradient boosting* (LightGBM, XGBoost, CatBoost) |
| :--- | :--- | :--- |
| **Dimensão dos dados** | Pequena a média. Limites indicados pelos autores: 10 000 linhas e 500 atributos no v2; 50 000 linhas e 2 000 atributos no 2.5. Para versões posteriores, consultar a documentação | Escalam a milhões de linhas |
| **Afinação** | Dispensa afinação para um bom resultado de partida | Os valores de fábrica raramente são os melhores; beneficiam de afinação e de engenharia de atributos |
| **Qualidade das probabilidades** | Em geral bem calibradas sem passos adicionais | Podem precisar de calibração posterior |
| **Hardware** | GPU recomendada | CPU comum é suficiente |
| **Latência** | Mais elevada: cada previsão processa todo o contexto de treino. Existem opções de destilação para modelos mais leves | Muito baixa, adequada a serviços em tempo real |
| **Licença** | Código aberto; pesos recentes só para uso não comercial; v2 com licença aberta | Totalmente abertas |
| **Explicabilidade e auditoria** | Menos ferramentas e menos prática estabelecida | Ecossistema maduro (importância de atributos, SHAP) |

**Regra prática.** Num problema novo com poucos milhares de linhas, o TabPFN é uma boa primeira tentativa, porque dá um resultado forte em minutos e serve de fasquia. Com dados grandes, requisitos de latência apertados, uso comercial sem licença ou necessidade de auditoria, as árvores continuam a ser a escolha natural.

**Nota sobre CPU.** Em CPU, o pacote `tabpfn` recusa por omissão conjuntos de treino com mais de 1 000 linhas, por causa do tempo de execução. Os datasets sugeridos abaixo ficam dentro desse limite.

## 13. Próximos passos

Basta alterar a secção 3 e voltar a correr o notebook a partir daí.

- **Outros datasets binários do OpenML:** `DATASET_NOME = "diabetes"`, `"blood-transfusion-service-center"` ou `"kc2"` (todos com `DATASET_VERSAO = 1`). A codificação do alvo adapta-se sozinha; se quiser escolher a classe positiva, defina `CLASSE_POSITIVA`.
- **Mais resolução estatística:** aumente `N_REPETICOES` para 5 ou 10.
- **Uma comparação mais exigente para o TabPFN:** aumente `N_ITER_AFINACAO` para 100 e veja se o LightGBM afinado se aproxima ou ultrapassa.
- **Outra versão do modelo:** troque `VERSAO_COM_TOKEN` na secção 2 (por exemplo, `ModelVersion.V2_5`) e compare gerações do TabPFN.
- **Decisão com custos:** a documentação do `credit-g` indica que classificar como bom um cliente mau custa cinco vezes mais do que o erro contrário. Escolha o limiar que minimiza esse custo, em vez de 0,5, e compare os modelos pelo custo total.

## 14. Referências e registo de alterações

**Referências**

- Hollmann et al., *Accurate predictions on small data with a tabular foundation model*, Nature, 2025 (TabPFN v2).
- Prior Labs, *TabPFN-2.5: Advancing the State of the Art in Tabular Foundation Models*, 2025: <https://arxiv.org/abs/2511.08667>
- Erickson et al., *TabArena: A Living Benchmark for Machine Learning on Tabular Data*, 2025.
- Nadeau e Bengio, *Inference for the Generalization Error*, Machine Learning, 2003 (teste t corrigido).
- Repositório e documentação do TabPFN: <https://github.com/PriorLabs/TabPFN> e <https://docs.priorlabs.ai/how-to-access-gated-models>

**O que mudou face à versão anterior deste notebook**

| Tema | Antes | Agora |
| :--- | :--- | :--- |
| Instalação | `tabpfn` sem versão; `seaborn` instalado sem ser usado | Versão fixa (`10.2.0`); só o necessário; versões impressas |
| Autenticação | Não era mencionada; a descarga dos pesos podia falhar | Secção própria, com token e alternativa aberta (v2) |
| Alvo | Fixo em `"good"`; trocar de dataset partia o notebook | Codificação genérica, com validação |
| Variáveis categóricas | Codificação ordinal igual para todos os modelos | Tratamento próprio por modelo, dentro de *pipelines* |
| Modelos de comparação | Só LightGBM com valores de fábrica | Referência, regressão logística, LightGBM de fábrica e afinado |
| Avaliação | Uma partição com 200 linhas de teste | Validação cruzada repetida, intervalos *bootstrap* e teste emparelhado corrigido |
| Métricas | ROC-AUC e *accuracy* | Mais *log loss*, Brier e *accuracy* equilibrada |
| Tempos | `fit` e `predict` somados; incluía a descarga dos pesos e duas passagens pelo modelo | Aquecimento prévio; `fit` e `predict` separados; uma só passagem |
| Conclusões | Escritas no texto antes de haver resultados | Geradas a partir dos números de cada execução |
| Conceitos | «Redes bayesianas»; limite de linhas vago | «Redes neuronais bayesianas»; limites por versão; nota sobre licenças |